# <center>
<div style="
    background: #f0f333ff;
    border-left: 5px solid #ecd242ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h1 style="
        text-align: center;
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 600;
    ">
    Exploration - Phase 1
    </h1>
</div>

##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Questions
    </h2>
</div>

La question centrale est : <br>**Parmi nos 10 000 numéros, lesquels sont valides, lesquels ne le sont pas — et lesquels n'ont pas pu être tranchés ?**

Questions préliminaires :

- Combien de pays distincts ? Combien de formats différents pour un même pays ?
- Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?
- Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?
- Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?
- Le CSV et l'Excel contiennent-ils vraiment la même chose une fois chargés ? Lequel prenez-vous comme source de vérité, et pourquoi ?
- Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Imports
    </h3>
</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Reading Data
df_csv = pd.read_csv('starter_kit/data/numeros_tva.csv', index_col='id', sep=',')
df_xl  = pd.read_excel('starter_kit/data/numeros_tva.xlsx', index_col='id')

In [2]:
df_csv.head(8)

,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie
id,,,,,
1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client
2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp
3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm
4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur
5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm
6,Kappa Logistics AB,DK,DK61530843,2025-01-14,import_fournisseur
7,Textil Services Oy,NL,NL505862176B88,2024-10-14,saisie_manuelle
8,Vertex Industries SARL,SE,SE.314125231601,2025-03-31,reprise_erp


##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Comparaison des deux fichiers
    </h2>
</div>

In [3]:
# Les noms de colonnes des 2 fichiers sont identiques
print(f'Colonnes du fichier CSV :\t {list(df_csv.columns)}')
print(f'Colonnes du fichier XL  :\t {list(df_xl.columns)}')

Colonnes du fichier CSV :	 ['raison_sociale', 'pays_declare', 'numero_tva', 'date_saisie', 'source_saisie']
Colonnes du fichier XL  :	 ['raison_sociale', 'pays_declare', 'numero_tva', 'date_saisie', 'source_saisie']


In [4]:
# Il y a le même nombre de lignes et de colonnes dans les 2 fichiers
print(f'Taille des données dans le fichier CSV :\t{df_csv.shape}')
print(f'Taille des données dans le fichier XL  :\t{df_csv.shape}')

Taille des données dans le fichier CSV :	(10000, 5)
Taille des données dans le fichier XL  :	(10000, 5)


In [5]:
# Les différences de données entre les 2 fichiers sont dans la colonne : "numero_tva"
print(f"Il y a {df_csv['numero_tva'].shape[0] - (df_csv['numero_tva'] == df_xl['numero_tva']).sum()} valeurs différentes entre les 2 fichiers")
(df_csv == df_xl).sum()

Il y a 146 valeurs différentes entre les 2 fichiers


raison_sociale    10000
pays_declare      10000
numero_tva         9854
date_saisie       10000
source_saisie     10000
dtype: int64

###
<div style="
    background: #9feaf2ff;
    border-left: 5px solid #1d28c1ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Etude de quelques données différentes
    </h3>
</div>

In [6]:
# Les différences de valeurs sont là où il y a des valeurs vides.
index_diff = df_csv.index[df_csv['numero_tva'] != df_xl['numero_tva']]
df_diff = pd.concat([df_csv.loc[index_diff, 'numero_tva'], df_xl.loc[index_diff, 'numero_tva']], axis=1)
display(df_diff.sample(10))
print(f"nombre de valeurs vides : {df_diff.isna().sum().sum()} sur un total de {df_diff.shape[0] * df_diff.shape[1]}")

,numero_tva,numero_tva
id,,
5419,NaN,NaN
4118,NaN,NaN
3161,NaN,NaN
9367,NaN,NaN
7698,NaN,NaN
4773,NaN,NaN
9724,NaN,NaN
5045,NaN,NaN
1696,NaN,NaN


nombre de valeurs vides : 292 sur un total de 292


**Conclusion :** Les seuls valeurs différentes entre les deux fichiers sont les valeurs vides donc les deux fichiers sont identiques, on peut étudier l'un ou l'autre comme on veut.

##
<div style="
    background: #40f0aa;
    border-left: 5px solid #0c7230ff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h2 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 500;
    ">
    Etude des pays
    </h2>
</div>

**Combien de pays distincts ? Combien de formats différents pour un même pays ?**

In [7]:
# Nombre de pays dans le jeu de données.
display(df_csv['pays_declare'].value_counts())
print(f'Il y a {df_csv['pays_declare'].nunique()} pays différents dans le jeu de données')

pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

Il y a 15 pays différents dans le jeu de données


In [8]:
df_fr = df_csv[df_csv['pays_declare'] == 'FR']['numero_tva'].to_frame(name='FR')
df_fr.shape

(974, 1)

In [ ]:
def fct(x):
    if pd.isna(x): return "Manquant"
    return "OK"

df_fr['test'] = df_fr.apply(fct, axis=1)
df_fr.head(100)

####
<div style="
    background: #eebbefff;
    border-left: 5px solid #d619efff;
    padding: 15px 25px;
    margin: 20px 0;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
">
    <h3 style="
        color: #2e3a59;
        font-family: 'Segoe UI', sans-serif;
        margin: 0;
        font-weight: 400;
    ">
    Etude de quelques données différentes
    </h3>
</div>